**[Course Home](../README.md) | [Python Lab](../python/01_knapsack_and_lagrangian_shadow_prices.ipynb) | [Prev: Ch 00: Overview](00_START_HERE.ipynb) | [Next: Ch 02: Submodular Set Functions](02_submodular_set_functions_and_diminishing_returns.ipynb)**

*Part I: Foundations of Resource Allocation & Portfolio Selection — Chapter 01 of 11*

---

# Chapter 1: Constrained Optimization — Knapsack Problems, LP Relaxations, Dual Lagrange Multipliers as Shadow Prices & Bang-for-Buck Ranking
### *Mathematical Formulations, Primal-Dual Economic Interpretation, Integrality Gaps, and the Frontiers of Resource Sizing in Distributed Systems*

---

## 1. Why Constrained Optimization Matters in Distributed Systems

In distributed software engineering, infrastructure operations, and continuous integration pipelines, resources are never infinite. Every engineering decision operates within hard physical and economic boundaries:

1. **Finite CI/CD Runner Fleets**: Cloud worker pools have finite core counts and memory quotas. A build-and-test merge queue must deliver verification feedback within strict service-level agreements (SLAs)—typically 15 to 30 minutes—to prevent pull request (PR) congestion and developer idle time.
2. **Hardware Testbeds**: Dedicated device lab testbeds (e.g., specialized microcontrollers, embedded boards, GPU inference clusters) cannot be scaled horizontally on demand. Execution involves physical flash cycles, reboot latencies, and thermal limits.
3. **Memory and Bandwidth Caps**: Containerized microservices operate under strict Linux cgroup memory limits. An in-memory cache or query aggregator cannot retain every candidate index or precomputed table without triggering Out-Of-Memory (OOM) evictions.
4. **Rate Limits and Egress Costs**: External API invocations, observability telemetry sinks, and distributed tracing spans face dollar budgets and network egress bandwidth ceilings.

Modern software repositories contain tens of thousands of automated tests, regression targets, static analysis passes, and diagnostic probes. Running the complete portfolio on every single git commit or PR is computationally impossible. An automated test suite that consumes 200 runner hours per commit across 500 daily PRs would require 100,000 runner hours per day, incurring unsustainable cloud infrastructure costs while paralyzing delivery velocity.

Therefore, infrastructure engineers face an inescapable combinatorial selection problem:

> **The Resource Allocation Dilemma**: Given a finite resource budget $B$ (measured in runner minutes, memory bytes, or hardware slots) and a pool of $n$ candidate execution targets—each characterized by a resource cost $c_i$ and an expected defect-detection or observability value $v_i$—which specific subset should be executed to maximize system reliability?

Constrained optimization provides the rigorous mathematical framework needed to solve this dilemma. Rather than relying on subjective heuristics ("run whatever the author modified" or "run a random 10% sample"), mathematical optimization defines precise trade-off frontiers, proves theoretical performance guarantees, and computes the exact economic value of adding more capacity.

---

## 2. The 0-1 Knapsack Problem: Formal Model & Complexity

The canonical baseline model for budget-constrained portfolio selection is the **0-1 Knapsack Problem**. 

### Formal Problem Statement

Let $\{1, 2, \dots, n\}$ denote a universe of $n$ discrete targets (such as CI/CD test suites or diagnostic checks). Each target $i$ possesses two deterministic attributes:
- A strictly positive **cost** $c_i > 0$, representing the required resource expenditure (e.g., execution time in minutes, memory in GiB, or compute cost in dollars).
- A strictly positive **value** $v_i > 0$, quantifying the expected utility or defect-detection power of executing target $i$ (e.g., historical fault catch rate, static code coverage weight, or risk score).

Let $B > 0$ denote the total available **budget** or capacity limit.

We define a binary decision vector $x = (x_1, x_2, \dots, x_n) \in \{0, 1\}^n$, where:

$$
x_i = \begin{cases} 1 & \text{if target } i \text{ is selected for execution}, \\ 0 & \text{if target } i \text{ is omitted}. \end{cases}
$$

The 0-1 Knapsack Problem is formulated as the integer linear program:

$$
\max_{x \in \{0, 1\}^n} \sum_{i=1}^n v_i x_i
$$

$$
\text{subject to} \quad \sum_{i=1}^n c_i x_i \le B
$$

### Combinatorial State Space Explosion

The feasible decision space consists of all binary vectors $x \in \{0, 1\}^n$ satisfying the linear budget constraint. In the unconstrained case, the total number of candidate subsets is:

$$
|\mathcal{P}(\{1, \dots, n\})| = 2^n
$$

As $n$ scales, brute-force enumeration becomes physically impossible:
- For $n = 30$, $2^{30} \approx 1.07 \times 10^9$ subsets (evaluable in several seconds on a modern CPU).
- For $n = 60$, $2^{60} \approx 1.15 \times 10^{18}$ subsets (decades of compute time across massive server clusters).
- For $n = 500$ (a typical mid-sized CI/CD test catalog), $2^{500} \approx 3.27 \times 10^{150}$ subsets, vastly exceeding the number of atoms in the observable universe.

### Computational Complexity: Weak NP-Hardness

The 0-1 Knapsack Problem is NP-hard. We establish this via a polynomial-time reduction from the **Subset Sum Problem**, a known NP-complete decision problem.

> **Theorem (NP-Hardness of 0-1 Knapsack)**: The 0-1 Knapsack decision problem is NP-complete.
>
> *Proof*: The decision version asks whether there exists a subset $S \subseteq \{1, \dots, n\}$ such that $\sum_{i \in S} c_i \le B$ and $\sum_{i \in S} v_i \ge K$.
> 1. **Membership in NP**: Given a certificate vector $x \in \{0, 1\}^n$, verifying both $\sum c_i x_i \le B$ and $\sum v_i x_i \ge K$ requires $O(n)$ arithmetic additions and comparisons, which runs in polynomial time.
> 2. **Reduction from Subset Sum**: An instance of Subset Sum consists of a set of positive integers $A = \{a_1, a_2, \dots, a_n\}$ and a target sum $T$. The question is whether any subset sums exactly to $T$. We construct an instance of the 0-1 Knapsack decision problem by setting $c_i = a_i$, $v_i = a_i$ for all $i \in \{1, \dots, n\}$, and choosing budget $B = T$ and target value $K = T$. If there exists a subset with total cost $\le T$ and total value $\ge T$, its sum must be exactly $T$ (since $c_i = v_i$). Thus, a polynomial-time algorithm for 0-1 Knapsack would solve Subset Sum in polynomial time. $\blacksquare$

Because the hardness of Subset Sum depends on the numerical magnitude of the values rather than purely combinatorial structure, 0-1 Knapsack is **weakly NP-hard**. When costs $c_i$ and budget $B$ are integers of moderate magnitude, the problem can be solved in pseudo-polynomial time via Dynamic Programming.

### The Indivisibility Principle

The fundamental friction in integer optimization is **indivisibility**: $x_i \in \{0, 1\}$. In the physical world of systems engineering:
- An integration test suite cannot be executed for "35% of its duration" and still provide valid pass/fail assertions.
- A physical worker node cannot be allocated as "0.4 of a server."
- A firmware binary cannot be flashed "halfway."

This indivisibility creates "packing waste"—slack capacity that cannot accommodate any remaining whole item—generating a gap between the theoretical continuous upper bound and the achievable discrete solution.

---

## 3. The Fractional LP Relaxation & Optimal Greedy Sorting

To analyze and approximate the intractable integer program, we relax the binary integrality constraint $x_i \in \{0, 1\}$ to the continuous unit interval $x_i \in [0, 1]$.

### Formulation of the Fractional Knapsack Problem

$$
\max_{x \in \mathbb{R}^n} \sum_{i=1}^n v_i x_i
$$

$$
\text{subject to} \quad \sum_{i=1}^n c_i x_i \le B, \quad 0 \le x_i \le 1 \quad \forall i \in \{1, \dots, n\}
$$

Here, $x_i$ represents the fraction of target $i$ selected. While fractional test execution is physically impossible in 0-1 selection, the fractional Linear Program (LP) provides:
1. An unassailable mathematical **upper bound** on the maximum possible discrete objective value: $V^*_{OPT} \le V^*_{LP}$.
2. A rigorous measure of marginal item efficiency.
3. The basis for dual shadow price calculations.

### The Efficiency Ratio (Bang-for-Buck Metric)

For each target $i$, define its **efficiency ratio** (also termed marginal density or bang-for-buck):

$$
r_i = \frac{v_i}{c_i}
$$

The ratio $r_i$ measures the value delivered per unit of resource consumed (e.g., defects detected per runner minute).

### Dantzig's Greedy Algorithm (1957)

George Dantzig proved that the continuous relaxation can be solved to global optimality by a single greedy pass over items sorted by efficiency ratio.

#### Algorithm Steps:
1. **Sort**: Index items in descending order of efficiency ratio:

$$r_1 \ge r_2 \ge r_3 \ge \dots \ge r_n$$

2. **Greedy Packing**: Iterate through sorted items $i = 1, 2, \dots, n$:
   - If cumulative cost does not exceed budget ($\sum_{j=1}^i c_j \le B$), set $x_i = 1$.
   - At the first item $s$ where $\sum_{j=1}^s c_j > B$, set:

$$x_s = \frac{B - \sum_{j=1}^{s-1} c_j}{c_s} \in [0, 1)$$

   - For all subsequent items $i > s$, set $x_i = 0$.
3. **Termination**: Return $x^* = (1, \dots, 1, x_s, 0, \dots, 0)$.

Item $s$ is designated the **split item** (or **critical item**).

### Formal Proof of Optimality for Dantzig's Greedy Rule

> **Theorem (Optimality of Fractional Greedy)**: The greedy solution $x^*$ produced by sorting items by descending efficiency ratio $r_i = v_i / c_i$ is globally optimal for the Fractional Knapsack LP.
>
> *Proof (Exchange Argument)*:
> Let $x^*$ be the greedy fractional solution with split item $s$. By construction, $x^*_i = 1$ for all $i < s$, $x^*_s \in [0, 1]$, and $x^*_i = 0$ for all $i > s$. Furthermore, $\sum_{i=1}^n c_i x^*_i = B$ (assuming $\sum c_i \ge B$).
>
> Let $y = (y_1, \dots, y_n)$ be any other feasible fractional solution satisfying $0 \le y_i \le 1$ and $\sum_{i=1}^n c_i y_i \le B$. We evaluate the value difference:
>
> $$
> \sum_{i=1}^n v_i x^*_i - \sum_{i=1}^n v_i y_i = \sum_{i=1}^n v_i (x^*_i - y_i)
> $$
>
> Multiply and divide each term by $c_i$:
>
> $$
> \sum_{i=1}^n v_i (x^*_i - y_i) = \sum_{i=1}^n r_i c_i (x^*_i - y_i)
> $$
>
> Now partition the items relative to the split item $s$:
> - For $i < s$: $x^*_i = 1 \ge y_i$, so $(x^*_i - y_i) \ge 0$, and by the sorting order, $r_i \ge r_s$. Thus:
>   $$r_i c_i (x^*_i - y_i) \ge r_s c_i (x^*_i - y_i)$$
> - For $i = s$: $r_s c_s (x^*_s - y_s) = r_s c_s (x^*_s - y_s)$.
> - For $i > s$: $x^*_i = 0 \le y_i$, so $(x^*_i - y_i) \le 0$, and by the sorting order, $r_i \le r_s$. Thus:
>   $$r_i c_i (x^*_i - y_i) \ge r_s c_i (x^*_i - y_i)$$
>
> Summing over all $i \in \{1, \dots, n\}$:
>
> $$
> \sum_{i=1}^n r_i c_i (x^*_i - y_i) \ge r_s \sum_{i=1}^n c_i (x^*_i - y_i) = r_s \left( \sum_{i=1}^n c_i x^*_i - \sum_{i=1}^n c_i y_i \right)
> $$
>
> Since $\sum c_i x^*_i = B$ and $\sum c_i y_i \le B$, the difference is non-negative:
>
> $$
> \sum_{i=1}^n c_i x^*_i - \sum_{i=1}^n c_i y_i = B - \sum_{i=1}^n c_i y_i \ge 0
> $$
>
> Because $r_s > 0$, it follows that:
>
> $$
> \sum_{i=1}^n v_i x^*_i - \sum_{i=1}^n v_i y_i \ge r_s (B - \sum_{i=1}^n c_i y_i) \ge 0 \implies \sum_{i=1}^n v_i x^*_i \ge \sum_{i=1}^n v_i y_i
> $$
>
> Hence, no feasible solution $y$ can achieve a strictly higher objective than $x^*$. $\blacksquare$

---

## 4. Duality Theory, Lagrange Multipliers, and Shadow Prices

The Linear Programming relaxation is not merely a bounding technique; it reveals the profound economic structure of resource allocation through **Lagrangian Duality**.

### The Lagrangian Function

Consider the primal Fractional Knapsack problem written in standard minimization form:

$$
\min_{x} -\sum_{i=1}^n v_i x_i \quad \text{subject to} \quad \sum_{i=1}^n c_i x_i - B \le 0, \quad x_i - 1 \le 0, \quad -x_i \le 0
$$

We introduce Lagrange multipliers:
- $\lambda \ge 0$ for the budget constraint $\sum c_i x_i \le B$.
- $\mu_i \ge 0$ for each upper bound $x_i \le 1$.
- $\nu_i \ge 0$ for each lower bound $x_i \ge 0$.

The Lagrangian function $L(x, \lambda, \mu, \nu)$ is:

$$
L(x, \lambda, \mu, \nu) = -\sum_{i=1}^n v_i x_i + \lambda \left(\sum_{i=1}^n c_i x_i - B\right) + \sum_{i=1}^n \mu_i (x_i - 1) - \sum_{i=1}^n \nu_i x_i
$$

Rearranging terms by grouping decision variables $x_i$:

$$
L(x, \lambda, \mu, \nu) = -\lambda B - \sum_{i=1}^n \mu_i + \sum_{i=1}^n \left( \lambda c_i + \mu_i - \nu_i - v_i \right) x_i
$$

### Karush-Kuhn-Tucker (KKT) Optimality Conditions

At the primal-dual optimal solution $(x^*, \lambda^*, \mu^*, \nu^*)$, the KKT conditions mandate:

1. **Stationarity**: $\frac{\partial L}{\partial x_i} = 0 \implies \lambda c_i + \mu_i - \nu_i - v_i = 0$, or:

$$v_i - \lambda c_i = \mu_i - \nu_i$$

2. **Primal Feasibility**: $\sum_{i=1}^n c_i x_i \le B$ and $0 \le x_i \le 1$ for all $i$.
3. **Dual Feasibility**: $\lambda \ge 0$, $\mu_i \ge 0$, $\nu_i \ge 0$.
4. **Complementary Slackness**:

$$\lambda \left( \sum_{i=1}^n c_i x_i - B \right) = 0$$

$$\mu_i (x_i - 1) = 0 \quad \forall i$$

$$\nu_i x_i = 0 \quad \forall i$$

### Economic Characterization of the Split Item

Complementary slackness dictates the relationship between the dual multiplier $\lambda^*$ and the primal decisions $x^*_i$:

- **If $x^*_i = 1$ (Item fully selected)**:
  Then $x^*_i > 0 \implies \nu^*_i = 0$. By stationarity, $\mu^*_i = v_i - \lambda^* c_i$. Since $\mu^*_i \ge 0$, we have:

$$\frac{v_i}{c_i} \ge \lambda^*$$

- **If $x^*_i = 0$ (Item completely rejected)**:
  Then $x^*_i < 1 \implies \mu^*_i = 0$. By stationarity, $\nu^*_i = \lambda^* c_i - v_i \ge 0$, which yields:

$$\frac{v_i}{c_i} \le \lambda^*$$

- **If $0 < x^*_s < 1$ (The Split Item)**:
  Since $x^*_s$ is strictly between 0 and 1, both $\mu^*_s = 0$ (from $x^*_s < 1$) and $\nu^*_s = 0$ (from $x^*_s > 0$). Stationarity immediately requires:

$$v_s - \lambda^* c_s = 0 \implies \lambda^* = \frac{v_s}{c_s} = r_s$$

> **Fundamental Duality Result**: The optimal Lagrange multiplier $\lambda^*$ associated with the budget constraint is precisely the efficiency ratio $r_s$ of the split item!

### Derivation of the Shadow Price: $\lambda^* = \frac{\partial V^*_{LP}}{\partial B}$

What is the operational meaning of $\lambda^*$ to a systems architect?

Let $V^*_{LP}(B)$ denote the optimal objective value of the Fractional Knapsack LP as a function of the available capacity $B$.

By strong duality for linear programs, the primal optimal value equals the dual optimal value:

$$
V^*_{LP}(B) = \min_{\lambda \ge 0, \mu \ge 0} \left[ \lambda B + \sum_{i=1}^n \mu_i \right] \quad \text{subject to } \mu_i \ge v_i - \lambda c_i
$$

For a budget $B$ where the split item $s$ satisfies $0 < x^*_s < 1$, consider an infinitesimal perturbation $dB > 0$. The set of fully packed items $\{1, \dots, s-1\}$ remains unchanged. The extra capacity $dB$ is immediately allocated to packing an additional fraction $dx_s = \frac{dB}{c_s}$ of the split item.

The resulting change in total value is:

$$
d V^*_{LP} = v_s dx_s = v_s \left( \frac{dB}{c_s} \right) = \left( \frac{v_s}{c_s} \right) dB = \lambda^* dB
$$

Taking the limit as $dB \to 0$:

$$
\frac{\partial V^*_{LP}}{\partial B} = \lambda^*
$$

This equality establishes that **$\lambda^*$ is the Shadow Price of capacity**. It represents the marginal rate of return—the additional defect detection value gained per additional runner minute of budget.

### Subgradients and Directional Derivatives at Breakpoints

What happens when capacity $B$ exactly matches the cumulative cost of the first $k$ items ($\sum_{j=1}^k c_j = B$)?

At such points, no item is partially packed: $x^*_1 = \dots = x^*_k = 1$ and $x^*_{k+1} = \dots = x^*_n = 0$. The function $V^*_{LP}(B)$ is piecewise linear and concave, with a "kink" (discontinuous slope) at this budget boundary.

At this boundary, the directional derivatives differ:
- **Left Derivative (contraction of budget)**: Reducing capacity by $dB$ forces shedding the lowest-ratio packed item ($k$), incurring marginal value loss:

$$\frac{\partial^- V^*_{LP}}{\partial B} = r_k = \frac{v_k}{c_k}$$

- **Right Derivative (expansion of budget)**: Increasing capacity by $dB$ allows packing the highest-ratio available item ($k+1$), yielding marginal value gain:

$$\frac{\partial^+ V^*_{LP}}{\partial B} = r_{k+1} = \frac{v_{k+1}}{c_{k+1}}$$

Because $r_k \ge r_{k+1}$, the function $V^*_{LP}(B)$ exhibits diminishing marginal returns with respect to capacity $B$. The **subdifferential** $\partial V^*_{LP}(B)$ at the kink is the entire closed interval:

$$
\partial V^*_{LP}(B) = [r_{k+1}, r_k]
$$

Standard primal-dual LP solvers (such as simplex or interior-point methods in `scipy.optimize.linprog`) return the right-hand dual price $\lambda^* = r_{k+1}$ because any infinitesimal positive slack allows item $k+1$ to enter the basis.

### Systems Engineering Application: Fleet Sizing and Autoscaling

Shadow prices provide the mathematical foundation for economic fleet sizing:
- Suppose renting cloud CI/CD runners costs $P_{\text{runner}} = \$0.05$ per runner minute.
- Suppose defect detection is economically modeled: catching a severity-1 bug prior to main-branch merge prevents production outages worth an estimated $P_{\text{defect}} = \$1.00$ per defect-weight unit.
- If current fleet capacity yields a shadow price $\lambda^* = 2.20$ defect units per minute, the marginal revenue of capacity is:

$$\text{Marginal Benefit} = \lambda^* \times P_{\text{defect}} = 2.20 \times \$1.00 = \$2.20/\text{minute}$$

- Because $\$2.20 > \$0.05$, expanding the runner pool is economically value-accretive! The fleet should continue expanding until the shadow price drops to the break-even threshold:

$$\lambda^* = \frac{P_{\text{runner}}}{P_{\text{defect}}} = 0.05$$

---

## 5. Integrality Gap & Failure Modes of Naive Greedy

In practice, engineers frequently employ the **Naive Greedy Heuristic** for 0-1 Knapsack:
1. Sort targets by descending efficiency ratio: $r_1 \ge r_2 \ge \dots \ge r_n$.
2. Iterate through items: if target $i$ fits within the remaining capacity, select it ($x_i = 1$) and decrement remaining budget; otherwise, skip target $i$ and continue down the sorted list.

While intuitive and fast ($O(n \log n)$), naive greedy suffers from catastrophic failure modes due to integer indivisibility.

### Pathological Counterexample: Arbitrarily Bad Performance

Consider a 2-item instance with capacity $B \gg 1$:
- **Item 1**: Cost $c_1 = 1$, Value $v_1 = 1 + \epsilon$ (where $\epsilon > 0$ is small). Efficiency ratio $r_1 = \frac{1 + \epsilon}{1} = 1 + \epsilon$.
- **Item 2**: Cost $c_2 = B$, Value $v_2 = B$. Efficiency ratio $r_2 = \frac{B}{B} = 1.0$.

#### Execution Trace:
1. The naive greedy algorithm compares ratios: $r_1 = 1 + \epsilon > r_2 = 1.0$.
2. It evaluates Item 1 first: $c_1 = 1 \le B$. Item 1 is selected.
3. Remaining budget is now $B - 1$.
4. It evaluates Item 2: $c_2 = B > B - 1$. Item 2 does not fit and is rejected.
5. Total value obtained by Naive Greedy:

$$v_{\text{greedy}} = 1 + \epsilon$$

#### The True Optimal Solution:
The optimal 0-1 solution selects Item 2 alone, since $c_2 = B \le B$:

$$
V^*_{OPT} = v_2 = B
$$

#### Approximation Ratio:
The competitive ratio of naive greedy is:

$$
\frac{v_{\text{greedy}}}{V^*_{OPT}} = \frac{1 + \epsilon}{B} \xrightarrow{B \to \infty} 0
$$

As budget $B$ grows, the fraction of optimal value captured by naive greedy approaches zero! A high-density trivial item blocks a massive, high-value target that would have utilized the entire runner fleet.

### The 2-Approximation Repair

Fortunately, this pathology can be completely cured by a remarkably simple algorithmic repair: **compare the greedy solution against the single best feasible item**.

#### Modified Greedy Algorithm:
1. Compute the naive greedy subset $S_{\text{greedy}}$ by ratio sorting and greedy packing. Let $v(S_{\text{greedy}}) = \sum_{i \in S_{\text{greedy}}} v_i$.
2. Find the single most valuable item that fits within the budget:

$$i^* = \arg\max_{i: c_i \le B} v_i$$

3. Return the solution with the larger value:

$$\hat{S} = \arg\max \Big( v(S_{\text{greedy}}), \; v_{i^*} \Big)$$

### Mathematical Proof of the 2-Approximation Bound

> **Theorem (2-Approximation Guarantee)**: The modified greedy algorithm achieves at least 50% of the optimal integer knapsack value:
>
> $$
> \max\Big( v(S_{\text{greedy}}), \; v_{i^*} \Big) \ge \frac{1}{2} V^*_{OPT}
> $$
>
> *Proof*:
> Consider the fractional LP relaxation with items sorted by efficiency ratio $r_1 \ge r_2 \ge \dots \ge r_n$.
> Let $s$ be the split item where cumulative cost exceeds $B$:
>
> $$
> \sum_{j=1}^{s-1} c_j \le B \quad \text{and} \quad \sum_{j=1}^s c_j > B
> $$
>
> The optimal fractional LP value is:
>
> $$
> V^*_{LP} = \sum_{j=1}^{s-1} v_j + x^*_s v_s
> $$
>
> Since $x^*_s < 1$ and $v_s > 0$:
>
> $$
> V^*_{LP} < \sum_{j=1}^{s-1} v_j + v_s
> $$
>
> Now evaluate the two components of our repaired algorithm:
> 1. By construction, the first $s-1$ items all fit within budget $B$. The naive greedy algorithm packs all items $1, \dots, s-1$ before encountering the split item. (It may also pack additional items after $s$). Therefore:
>    $$v(S_{\text{greedy}}) \ge \sum_{j=1}^{s-1} v_j$$
> 2. The split item $s$ satisfies $c_s \le B$ (if $c_s > B$, it could never be part of any feasible fractional pack). By definition, $v_{i^*}$ is the maximum value of any single feasible item, so:
>    $$v_{i^*} \ge v_s$$
>
> Summing these two inequalities:
>
> $$
> v(S_{\text{greedy}}) + v_{i^*} \ge \sum_{j=1}^{s-1} v_j + v_s > V^*_{LP}
> $$
>
> Because the optimal integer solution cannot exceed the fractional relaxation ($V^*_{OPT} \le V^*_{LP}$):
>
> $$
> v(S_{\text{greedy}}) + v_{i^*} > V^*_{OPT}
> $$
>
> For any two positive real numbers, the maximum is at least their arithmetic mean:
>
> $$
> \max\Big( v(S_{\text{greedy}}), \; v_{i^*} \Big) \ge \frac{1}{2} \Big( v(S_{\text{greedy}}) + v_{i^*} \Big) > \frac{1}{2} V^*_{OPT}
> $$
>
> Dividing by $V^*_{OPT}$:
>
> $$
> \frac{\max( v(S_{\text{greedy}}), v_{i^*} )}{V^*_{OPT}} \ge \frac{1}{2}
> $$
>
> This completes the proof. $\blacksquare$

The modified algorithm guarantees a **worst-case approximation ratio of 2** (meaning it captures at least 50% of the true mathematical optimum), while executing in $O(n \log n)$ time due to sorting.

---

## 6. Dynamic Programming for Exact 0-1 Knapsack

When the budget $B$ is an integer of moderate magnitude (e.g., $B \le 10^5$), the exact global optimum can be computed in pseudo-polynomial time using **Dynamic Programming (DP)**.

### Bellman's Principle of Optimality

Dynamic programming decomposes the global decision into a sequence of nested subproblems. Let $V(i, w)$ denote the optimal value achievable using a subset of the first $i$ items $\{1, \dots, i\}$ with an available capacity limit of $w \in \{0, 1, \dots, B\}$.

For item $i$ with cost $c_i$ and value $v_i$, we face a binary decision:
1. **Exclude item $i$**: The optimal value is simply the best value achievable using the first $i-1$ items with capacity $w$:

$$V(i, w) = V(i-1, w)$$

2. **Include item $i$** (valid only if $c_i \le w$): The optimal value is the value of item $i$ plus the best value achievable using the first $i-1$ items with the remaining capacity $w - c_i$:

$$V(i, w) = V(i-1, w - c_i) + v_i$$

### The Bellman Recurrence Relation

$$
V(i, w) = \begin{cases}
V(i-1, w) & \text{if } c_i > w \\
\max\Big( V(i-1, w), \; V(i-1, w - c_i) + v_i \Big) & \text{if } c_i \le w
\end{cases}
$$

#### Base Cases:
- $V(0, w) = 0$ for all $w \in \{0, \dots, B\}$ (zero items yield zero value).
- $V(i, 0) = 0$ for all $i \in \{0, \dots, n\}$ (zero capacity accommodates zero value).

### State Space & Computational Complexity

- **Time Complexity**: The algorithm computes a 2D table of size $(n+1) \times (B+1)$. Each table entry $V(i, w)$ requires $O(1)$ operations (one comparison and one addition). Thus, total runtime is:

$$T(n, B) = O(n B)$$

- **Space Complexity**:
  - Storing the full 2D table requires $O(n B)$ space.
  - If only the optimal value is needed, space can be reduced to $O(B)$ by using a 1D array and updating capacity $w$ in reverse order from $B$ down to $c_i$ (to prevent an item from being counted multiple times).

### Why $O(nB)$ is Pseudo-Polynomial

In theoretical computer science, an algorithm is *polynomial* only if its runtime is bounded by a polynomial in the length of the binary input encoding.
- The input to knapsack consists of $n$ numbers plus budget $B$.
- The number of bits required to encode integer $B$ is $k = \lceil \log_2 B \rceil$.
- Thus, $B = 2^k$.
- The runtime $O(n B) = O(n 2^k)$ is **exponential** in the number of bits $k$.

If budget $B$ is a 64-bit integer ($B \approx 10^{18}$), the DP table would require billions of gigabytes of memory and centuries to compute. However, in distributed systems engineering, budgets are typically measured in discrete seconds, runner slots, or integer megabytes (e.g., $B = 60$ minutes, $B = 1800$ seconds, or $B = 1024$ MiB), making DP blazingly fast in practice ($< 5$ milliseconds).

---

## 7. Step-by-Step Worked Numerical Example with Real Numbers

To crystallize the mathematical derivations, let us solve a concrete systems engineering resource allocation problem from scratch.

### Problem Instance: CI/CD Integration Test Selection

Suppose a distributed continuous integration system has a strict merge queue budget of **$B = 50$ runner minutes**. A pull request touches multiple subsystems, triggering **$n = 6$ candidate integration test targets**.

The measured execution costs (in minutes) and historical defect-detection values (quantified defect risk scores) are:

| Target ID ($i$) | Target Name | Cost $c_i$ (min) | Value $v_i$ (score) | Efficiency Ratio $r_i = v_i / c_i$ |
| :---: | :--- | :---: | :---: | :---: |
| **1** | Unit & Lint Suite | 5 | 12 | $12 / 5 = \mathbf{2.400}$ |
| **2** | Storage Driver Probe | 10 | 30 | $30 / 10 = \mathbf{3.000}$ |
| **3** | Network RPC Regression | 15 | 40 | $40 / 15 = \mathbf{2.667}$ |
| **4** | Security Boundary Check | 20 | 48 | $48 / 20 = \mathbf{2.400}$ |
| **5** | E2E Cluster Integration | 25 | 55 | $55 / 25 = \mathbf{2.200}$ |
| **6** | Soak & Concurrency Test | 30 | 60 | $60 / 30 = \mathbf{2.000}$ |

Total cost of all 6 targets is $5 + 10 + 15 + 20 + 25 + 30 = 105$ minutes, far exceeding our budget of $B = 50$ minutes.

---

### Step 1: Sorting by Efficiency Ratio

We sort the targets in descending order of efficiency ratio $r_i$:

1. **Target 2**: $r_2 = 3.000$ ($c_2 = 10, v_2 = 30$)
2. **Target 3**: $r_3 = 2.667$ ($c_3 = 15, v_3 = 40$)
3. **Target 1**: $r_1 = 2.400$ ($c_1 = 5, v_1 = 12$)
4. **Target 4**: $r_4 = 2.400$ ($c_4 = 20, v_4 = 48$)
5. **Target 5**: $r_5 = 2.200$ ($c_5 = 25, v_5 = 55$)
6. **Target 6**: $r_6 = 2.000$ ($c_6 = 30, v_6 = 60$)

*(Note: Target 1 and Target 4 tie at $r = 2.400$. We break the tie by lower cost: Target 1 first).*

---

### Step 2: Fractional LP Relaxation Solution

We execute Dantzig's greedy fractional packing:

- **Pack Target 2**: Cost = 10 min. Remaining budget = $50 - 10 = 40$ min. Cumulative value = 30.
- **Pack Target 3**: Cost = 15 min. Remaining budget = $40 - 15 = 25$ min. Cumulative value = $30 + 40 = 70$.
- **Pack Target 1**: Cost = 5 min. Remaining budget = $25 - 5 = 20$ min. Cumulative value = $70 + 12 = 82$.
- **Pack Target 4**: Cost = 20 min. Remaining budget = $20 - 20 = 0$ min. Cumulative value = $82 + 48 = 130$.

The budget $B = 50$ minutes is now completely exhausted.
- **Split Item**: Target 5 is the next candidate with $r_5 = 2.200$. Because the remaining capacity is exactly 0, $x^*_5 = 0 / 25 = 0.0$.
- **Optimal Fractional Vector**:

$$x^* = (x_1^*, x_2^*, x_3^*, x_4^*, x_5^*, x_6^*) = (1.0, 1.0, 1.0, 1.0, 0.0, 0.0)$$

- **LP Optimal Value**:

$$V^*_{LP} = 12(1) + 30(1) + 40(1) + 48(1) + 55(0) + 60(0) = \mathbf{130.0}$$

Because all decision variables $x^*_i$ happen to be binary ($1$ or $0$), this fractional LP solution is **naturally integral**! 

---

### Step 3: Exact Dual Shadow Price Extraction

What is the shadow price $\lambda^*$ at $B = 50$?

- If capacity increases by $\Delta B$ (for $\Delta B \le 25$), the solver packs a fraction of the next available target, **Target 5** ($r_5 = 2.200$). The right-hand derivative is:

$$\lambda^* = \frac{\partial^+ V^*_{LP}}{\partial B} = \frac{v_5}{c_5} = \frac{55}{25} = \mathbf{2.200}$$

- If capacity decreases by $\Delta B$ (for $\Delta B \le 20$), the solver sheds a fraction of the least efficient packed target, **Target 4** (or Target 1, both having $r = 2.400$). The left-hand derivative is:

$$\frac{\partial^- V^*_{LP}}{\partial B} = \mathbf{2.400}$$

At the exact integer coordinate $B = 50$, the subdifferential is $[2.200, 2.400]$. Solvers like `scipy.optimize.linprog(method='highs')` report the dual variable associated with the budget constraint as **$\lambda^* = 2.200$**, which represents the marginal value of capacity expansion.

---

### Step 4: Full Dynamic Programming Table

Let us construct the complete DP table $V(i, w)$ for $i \in \{1, \dots, 6\}$ across capacity increments of 5 minutes ($w \in \{0, 5, 10, \dots, 50\}$).

The recurrence is:

$$V(i, w) = \max\Big( V(i-1, w), \; V(i-1, w - c_i) + v_i \Big)$$

#### Full Numerical DP Matrix:

| $i$ | Target ($c_i, v_i$) | $w=0$ | $w=5$ | $w=10$ | $w=15$ | $w=20$ | $w=25$ | $w=30$ | $w=35$ | $w=40$ | $w=45$ | $w=50$ |
| :---: | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **0** | Baseline (Empty) | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| **1** | Target 1 ($5, 12$) | 0 | 12 | 12 | 12 | 12 | 12 | 12 | 12 | 12 | 12 | 12 |
| **2** | Target 2 ($10, 30$) | 0 | 12 | 30 | 42 | 42 | 42 | 42 | 42 | 42 | 42 | 42 |
| **3** | Target 3 ($15, 40$) | 0 | 12 | 30 | 42 | 52 | 70 | 82 | 82 | 82 | 82 | 82 |
| **4** | Target 4 ($20, 48$) | 0 | 12 | 30 | 42 | 52 | 70 | 82 | 90 | 100 | 118 | **130** |
| **5** | Target 5 ($25, 55$) | 0 | 12 | 30 | 42 | 52 | 70 | 82 | 90 | 100 | 118 | **130** |
| **6** | Target 6 ($30, 60$) | 0 | 12 | 30 | 42 | 52 | 70 | 82 | 90 | 100 | 118 | **130** |

#### Backtracking to Recover the Optimal Set:
Starting at cell $V(6, 50) = 130$:
- Compare $V(6, 50) = 130$ with $V(5, 50) = 130$: Values match $\implies$ **Omit Target 6**.
- Compare $V(5, 50) = 130$ with $V(4, 50) = 130$: Values match $\implies$ **Omit Target 5**.
- Compare $V(4, 50) = 130$ with $V(3, 50) = 82$: Values differ! $\implies$ **Include Target 4** ($c_4 = 20, v_4 = 48$). Remaining capacity $w = 50 - 20 = 30$.
- Compare $V(3, 30) = 82$ with $V(2, 30) = 42$: Values differ! $\implies$ **Include Target 3** ($c_3 = 15, v_3 = 40$). Remaining capacity $w = 30 - 15 = 15$.
- Compare $V(2, 15) = 42$ with $V(1, 15) = 12$: Values differ! $\implies$ **Include Target 2** ($c_2 = 10, v_2 = 30$). Remaining capacity $w = 15 - 10 = 5$.
- Compare $V(1, 5) = 12$ with $V(0, 5) = 0$: Values differ! $\implies$ **Include Target 1** ($c_1 = 5, v_1 = 12$). Remaining capacity $w = 5 - 5 = 0$.

**Optimal Selected Set**: $\{\text{Target 1}, \text{Target 2}, \text{Target 3}, \text{Target 4}\}$.
- Total Execution Cost: $5 + 10 + 15 + 20 = \mathbf{50\text{ minutes}}$ (100% capacity utilization).
- Total Defect Score: $12 + 30 + 40 + 48 = \mathbf{130.0}$.

---

### Step 5: Side-by-Side Algorithm Comparison

| Optimization Method | Selected Targets | Total Cost | Total Value | Optimality Ratio |
| :--- | :---: | :---: | :---: | :---: |
| **Fractional LP Relaxation** | $\{1, 2, 3, 4\}$ | 50.0 min | **130.0** | 100.0% (Upper Bound) |
| **Exact 0-1 Dynamic Programming** | $\{1, 2, 3, 4\}$ | 50.0 min | **130.0** | **100.0% (Optimal)** |
| **Naive Greedy Heuristic** | $\{2, 3, 1, 4\}$ | 50.0 min | **130.0** | 100.0% |
| **Single Best Feasible Target** | $\{6\}$ | 30.0 min | 60.0 | 46.2% |
| **2-Approximation Greedy** | $\{2, 3, 1, 4\}$ | 50.0 min | **130.0** | 100.0% |

At $B = 50$, the integrality gap is exactly zero: $V^*_{LP} - V^*_{OPT} = 130 - 130 = 0$.

---

### Step 6: Sensitivity Analysis — What Happens When Budget Shifts?

In production, merge queue budgets fluctuate. What happens if runner pool constraints change?

#### Case A: Reduced Budget $B = 40$ minutes
- **Fractional LP**: Packs Targets 2, 3, 1 (cost = 30 min, value = 82). Remaining capacity = 10 min. Split item is Target 4 ($c_4 = 20$). It takes fraction $x_4 = 10/20 = 0.5$, adding $0.5 \times 48 = 24$.

$$V^*_{LP}(40) = 82 + 24 = \mathbf{106.0}$$

- **Exact DP**: Looking at the DP table column $w = 40$:

$$V(6, 40) = \mathbf{100.0}$$

  Backtracking reveals optimal set $\{\text{Target 1}, \text{Target 3}, \text{Target 4}\}$ with cost $5 + 15 + 20 = 40$ min and value $12 + 40 + 48 = 100$.
- **Naive Greedy**: Evaluates in sorted order $\{2, 3, 1, 4, 5, 6\}$.
  Packs Target 2 (10 min), Target 3 (15 min), Target 1 (5 min). Cumulative cost = 30 min, remaining = 10 min.
  Target 4 (cost 20) does not fit! Target 5 (cost 25) does not fit! Target 6 (cost 30) does not fit!

$$v_{\text{greedy}} = 30 + 40 + 12 = \mathbf{82.0}$$

> **Key Observation at $B = 40$**:
> - Integrality Gap: $V^*_{LP} - V^*_{OPT} = 106.0 - 100.0 = \mathbf{6.0}$ ($106/100 = 1.060$).
> - Naive Greedy Shortfall: DP achieves 100, whereas Naive Greedy gets only 82 (an 18% deficit!).
> - Naive greedy was trapped by picking Target 2 early, leaving 10 minutes of unusable slack capacity. The exact DP recognized that swapping Target 2 for Target 4 produced a superior packing.

#### Case B: Budget $B = 45$ minutes
- **Fractional LP**: Packs Targets 2, 3, 1 (cost 30 min, value 82). Remaining = 15 min. Split item Target 4 takes $x_4 = 15/20 = 0.75$, adding $0.75 \times 48 = 36$.

$$V^*_{LP}(45) = 82 + 36 = \mathbf{118.0}$$

- **Exact DP**: Column $w = 45$ shows $V(6, 45) = \mathbf{118.0}$.
  Backtracking yields $\{\text{Target 2}, \text{Target 3}, \text{Target 4}\}$ with cost $10 + 15 + 20 = 45$ min and value $30 + 40 + 48 = 118$.
- **Naive Greedy**: Packs Targets 2, 3, 1 (cost 30 min, value 82). Remaining = 15 min. Target 4 (20 min) fails to fit! Value remains stuck at **82.0**.
  Here, exact DP beats naive greedy by $118 - 82 = \mathbf{36\text{ points}}$ (+43.9% gain)!

---

## 8. Systems Engineering Synthesis & Bridge to Submodularity

### The Systems Engineer's Knapsack Decision Matrix

When architecting a production test selection or resource allocation service, how should an engineer choose between these methods?

| Metric / Dimension | Fractional LP Relaxation | 2-Approximation Greedy | Exact Dynamic Programming |
| :--- | :--- | :--- | :--- |
| **Computational Complexity** | $O(n \log n)$ via Dantzig sorting | $O(n \log n)$ | $O(n B)$ pseudo-polynomial |
| **Execution Latency** | $< 1$ millisecond | $< 0.1$ millisecond | $1 - 100$ milliseconds |
| **Solution Guarantee** | Fractional upper bound: $V^*_{LP} \ge V^*_{OPT}$ | Proven $\ge 50\%$ of $V^*_{OPT}$ (often $> 95\%$ in practice) | Exact mathematical optimum: $V^*_{OPT}$ |
| **Best Production Use Case** | Pruning branches in Branch-and-Bound; calculating shadow prices for capacity sizing | Real-time / online merge queues with strict sub-millisecond dispatch deadlines | Offline regression suite planning with discrete, integer budgets ($B \le 10^5$) |

### The Core Structural Assumption of the Knapsack Model

The 0-1 Knapsack formulation rests upon a foundational mathematical assumption: **modularity (linear additivity)**.

$$
V(S) = \sum_{i \in S} v_i
$$

Under this assumption:
- Each target $i$ delivers a fixed, independent value $v_i$, completely invariant to which other targets have already been executed.
- The marginal value of adding target $i$ to an empty set is identical to adding target $i$ to a set that already contains 50 other targets:

$$\Delta(i \mid S) = V(S \cup \{i\}) - V(S) = v_i \quad \forall S$$

### Why Modularity Fails in Real-World Systems

In distributed software systems, hardware device testing, and observability infrastructure, **items are rarely independent**:

1. **Test Coverage Overlap**: If Target A executes the user authentication module, and Target B is an end-to-end checkout test that also passes through the authentication module, executing both yields redundant verification. The marginal defect-detection value of Target B is sharply reduced once Target A has already executed.
2. **Sensor & Probe Redundancy**: In a fleet of 10,000 servers, query latency anomalies observed by sensor $s_1$ heavily correlate with anomalies captured by neighboring sensor $s_2$ on the same network rack.
3. **Observability Saturation**: The first few distributed tracing spans provide immense architectural insight; the ten-thousandth span mostly confirms what the system already knows.

In all these environments, the marginal utility of adding an element **diminishes as the set of already selected elements grows**:

$$
V(S \cup \{i\}) - V(S) \le V(T \cup \{i\}) - V(T) \quad \text{whenever } T \subseteq S
$$

This fundamental mathematical property—the discrete equivalent of concavity—is called **Submodularity**.

In **Chapter 02**, we leave behind the additive knapsack model and step into the richer, more realistic world of **Submodular Optimization**, where we will define set functions, formalize diminishing returns, and discover how greedy algorithms achieve provable $(1 - 1/e)$ approximation guarantees across complex distributed infrastructures.

---

**[Next: Chapter 02 — Submodular Set Functions & Diminishing Returns >>](02_submodular_set_functions_and_diminishing_returns.ipynb)**
